# Bài 12 · Trực quan hoá cơ bản

**Lập trình xử lý dữ liệu (LTXLDL) · 2627-1 · Viện TTNT, UET-VNU**

> 💡 File → **Save a copy in Drive** trước khi sửa.

**Mục tiêu buổi học** — sau khi hoàn thành notebook, bạn sẽ:

1. Chọn dạng biểu đồ theo **câu hỏi**: đường, cột, histogram hoặc phân tán; vẽ bằng `fig, ax`.
2. Hoàn thiện một biểu đồ tự giải thích được: tiêu đề nêu thông điệp, trục có nhãn và đơn vị, có nguồn dữ liệu.
3. Xuất hình từ pipeline (`savefig` vào `figures/`) với phong cách đồng bộ (`rcParams`).
4. Nhận diện những cách trình bày dễ gây hiểu sai, như trục bị cắt hoặc các hình dùng thang đo khác nhau.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Khai báo phong cách một lần để dùng chung cho toàn bộ notebook
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25,
    "font.size": 12,
})
XANH, CAM = "#1E93AB", "#E8890C"
Path("figures").mkdir(exist_ok=True)

BASE = "https://data.insideairbnb.com/chile/rm/santiago/2026-06-29"
df = pd.read_csv(f"{BASE}/visualisations/listings.csv")
rv = pd.read_csv(f"{BASE}/visualisations/reviews.csv", parse_dates=["date"])
print(df.shape, rv.shape)

## 1. Biểu đồ đường — diễn biến theo thời gian

In [ ]:
# Tháng 06/2026 chưa đủ dữ liệu: mốc chụp là 29/06 và đánh giá có thể được đăng muộn
thang = (rv[rv["date"] < "2026-06-01"]
         .set_index("date").resample("ME").size().loc["2023":])   # bỏ kỳ chưa đủ dữ liệu

fig, ax = plt.subplots(figsize=(9, 3.8))
ax.plot(thang.index, thang.values, color=XANH, lw=2)
ax.set_title("Thị trường Santiago phục hồi và tăng tốc sau 2023", loc="left", fontweight="bold")
ax.set_ylabel("số đánh giá / tháng")
ax.text(1, -0.18, "Nguồn: Inside Airbnb, 29/06/2026", transform=ax.transAxes,
        ha="right", fontsize=9, color="#777")
fig.savefig("figures/reviews_theo_thang.png", dpi=150, bbox_inches="tight")
plt.show()

Ba thành phần giúp hình tự giải thích được: tiêu đề nêu **thông điệp** thay vì chỉ gọi tên biểu đồ,
trục y có **đơn vị**, và cuối hình ghi **nguồn cùng ngày chụp dữ liệu**.

## 2. Biểu đồ thanh ngang — so sánh nhóm

In [ ]:
tk = df.groupby("neighbourhood")["price"].agg(median="median", n="size")
top = tk[tk["n"] >= 500].nlargest(8, "median").sort_values("median")

fig, ax = plt.subplots(figsize=(8.6, 4))
mau = [CAM if q == top["median"].idxmax() else XANH for q in top.index]
bars = ax.barh(top.index, top["median"], color=mau, height=0.6)
ax.bar_label(bars, [f" {v/1000:,.0f}k" for v in top["median"]], fontsize=10)
ax.set_title("Lo Barnechea bỏ xa phần còn lại về giá trung vị", loc="left", fontweight="bold")
ax.set_xlabel("giá trung vị (CLP/đêm), quận có ≥ 500 chỗ ở")
ax.grid(axis="y", alpha=0)
fig.savefig("figures/gia_theo_quan.png", dpi=150, bbox_inches="tight")
plt.show()

Thanh ngang giúp đọc tên quận dài mà không phải xoay chữ. Chỉ một thanh được tô cam để nhấn
nhóm cần chú ý; các màu còn lại giữ nhất quán để tránh tạo tín hiệu không cần thiết.

## 3. Histogram — hình dạng phân phối

In [ ]:
gia = df.loc[df["price"] > 0, "price"]

fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
axes[0].hist(gia, bins=55, color=XANH)
axes[0].set_title("Thang thường: chỉ thấy một cột", fontsize=11)
axes[1].hist(np.log10(gia), bins=55, color=CAM)
axes[1].set_title("Thang log10: thấy cả phân phối", fontsize=11)
for ax in axes:
    ax.set_ylabel("số chỗ ở")
axes[1].set_xlabel("log10(giá)")
plt.tight_layout(); plt.show()

## 4. Biểu đồ phân tán — cấu trúc theo vị trí

In [ ]:
s = df.dropna(subset=["price"]).sample(6000, random_state=1)
mau = np.where(s["price"] > s["price"].median(), CAM, XANH)

fig, ax = plt.subplots(figsize=(8, 4.4))
ax.scatter(s["longitude"], s["latitude"], s=6, c=mau, alpha=0.45, linewidths=0)
ax.set_aspect("equal")
ax.set_title("Nửa đắt của thị trường (cam) dồn về đông bắc", loc="left", fontweight="bold")
ax.set_xlabel("kinh độ"); ax.set_ylabel("vĩ độ")
plt.tight_layout(); plt.show()

Kinh độ, vĩ độ và hai nhóm màu đã cho thấy cấu trúc không gian. Chấm nhỏ cùng `alpha=0.45`
giúp hạn chế hiện tượng 6.000 điểm che lấp nhau. Bài 13 sẽ bổ sung ranh giới quận.

## 5. Xuất hình — PNG, SVG hay PDF?

`savefig` chọn định dạng theo **đuôi file**:

- **Raster** (`PNG`): lưới điểm ảnh, hợp với hình dày điểm hoặc ảnh chụp; `dpi` chỉ có nghĩa ở đây.
- **Vector** (`SVG`, `PDF`): đường nét, phóng to vẫn nét — `SVG` cho web, `PDF` cho báo cáo và in ấn.

Ảnh vector nét hơn nhưng mỗi điểm là một đối tượng, nên hình quá dày điểm làm file `SVG` phình rất to. Hãy tự lưu một hình thưa điểm và một hình dày điểm ra cả ba định dạng rồi so dung lượng:

In [ ]:
import os

# Hai hình để so sánh: thưa điểm (đường) và dày điểm (bản đồ ~18k điểm)
fig_thua, ax = plt.subplots(figsize=(8.8, 3.6))
ax.plot(thang.index, thang.values, color=XANH, lw=2)

diem = df.dropna(subset=["price"])                    # toàn bộ chỗ ở có giá
mau = np.where(diem["price"] > diem["price"].median(), CAM, XANH)
fig_day, ax = plt.subplots(figsize=(8.6, 4.6))
ax.scatter(diem["longitude"], diem["latitude"], s=6, c=mau, alpha=0.45, linewidths=0)
plt.close("all")                                      # chỉ đo dung lượng, không cần hiện hình

def kich_thuoc(fig, ten):
    for duoi in ["png", "svg", "pdf"]:                # matplotlib chọn định dạng theo đuôi file
        fig.savefig(f"figures/{ten}.{duoi}", dpi=150)
    def doc(duoi):
        kb = os.path.getsize(f"figures/{ten}.{duoi}") / 1024
        return f"{kb/1024:.1f} MB" if kb >= 1024 else f"{kb:.0f} KB"
    print(f"{ten:8}  PNG {doc('png'):>8}  SVG {doc('svg'):>8}  PDF {doc('pdf'):>8}")

kich_thuoc(fig_thua, "duong")    # thưa điểm -> vector (SVG/PDF) nhẹ và nét
kich_thuoc(fig_day,  "ban_do")   # dày điểm  -> PNG nhẹ hơn SVG hàng chục lần

## 6. Biểu đồ có thể gây hiểu sai như thế nào?

In [ ]:
gia_loai = df.groupby("room_type")["price"].median() / 1000
gia_loai = gia_loai[["Private room", "Entire home/apt", "Hotel room"]]

fig, axes = plt.subplots(1, 2, figsize=(9.6, 3.5))
for ax, y0, ten, mau_t in [(axes[0], 30, "Trục cắt từ 30 — chênh lệch bị thổi phồng", "#c0392b"),
                            (axes[1], 0, "Trục từ 0 — tỷ lệ thật", "#2E8B57")]:
    bars = ax.bar(["Phòng riêng", "Nguyên căn", "Khách sạn"], gia_loai.values, color=XANH, width=0.55)
    ax.set_ylim(y0, 130)
    ax.set_title(ten, color=mau_t, fontsize=11, fontweight="bold")
    ax.set_ylabel("giá trung vị (nghìn CLP)")
    ax.grid(axis="x", alpha=0)
plt.tight_layout(); plt.show()

Cùng một bộ số liệu nhưng hai cách đặt trục tạo ra ấn tượng khác nhau. Với **biểu đồ cột, trục y
cần bắt đầu từ 0** vì chiều cao biểu diễn độ lớn. Biểu đồ đường có thể thu hẹp trục để làm rõ biến thiên.

## 7. Bài tập tại lớp

### Bài 1 — Từ nháp đến xuất bản

Cell dưới tạo một bản nháp bằng một dòng lệnh. Hãy hoàn thiện bằng `fig, ax`, tiêu đề nêu thông điệp,
nhãn trục có đơn vị, nguồn dữ liệu và lưu vào `figures/`. Sau cùng, kiểm tra xem người đọc có thể
hiểu đúng hình mà không cần dựa vào tiêu đề hay không.

In [ ]:
# Bản nháp:
df["room_type"].value_counts().plot.barh()
plt.show()

# TODO: bản xuất bản của bạn
ty_le = df["room_type"].value_counts(normalize=True).sort_values() * 100
fig, ax = plt.subplots(figsize=(8, 3.2))
bars = ax.barh(ty_le.index, ty_le.values, color=XANH, height=0.55)
ax.bar_label(bars, [f" {v:.0f}%" for v in ty_le.values])
ax.set_title("4 trên 5 chỗ ở tại Santiago là nguyên căn", loc="left", fontweight="bold")
ax.set_xlabel("% tổng số chỗ ở (n = 18.534)")
ax.grid(axis="y", alpha=0)
fig.savefig("figures/ty_le_loai_phong.png", dpi=150, bbox_inches="tight")
plt.show()

### Bài 2 — So sánh hai mốc chụp trên cùng thang đo

Tải thêm bản 09/2025 (`{BASE_T9}/visualisations/listings.csv` với
`BASE_T9 = ".../santiago/2025-09-27"`). Vẽ hai histogram giá trên thang log10 cho hai mốc chụp với
**cùng thang trục** (`sharex=True, sharey=True`). Phân phối giá có dịch chuyển không?

In [ ]:
# TODO Bài 2:
t9 = pd.read_csv("https://data.insideairbnb.com/chile/rm/santiago/"
                 "2025-09-27/visualisations/listings.csv")
g9 = t9.loc[t9["price"] > 0, "price"]

fig, axes = plt.subplots(1, 2, figsize=(10, 3.4), sharex=True, sharey=True)
axes[0].hist(np.log10(g9), bins=50, color=XANH)
axes[0].set_title("09/2025")
axes[1].hist(np.log10(gia), bins=50, color=XANH)
axes[1].set_title("06/2026")
fig.suptitle("Hình dáng giữ nguyên nhưng cả phân phối dời phải — giá trung vị +45% sau 9 tháng", fontweight="bold")
plt.tight_layout(); plt.show()

### Bài 3 — Chọn dạng cho 3 câu hỏi

Với mỗi câu hỏi, hãy chọn dạng biểu đồ và vẽ bằng không quá 8 dòng mã nguồn cho mỗi hình:

1. "Số chỗ ở giữa 10 quận đứng đầu chênh lệch như thế nào?"
2. "Chỉ số `reviews_per_month` phân bố ra sao (đa số im ắng hay đều đặn)?"
3. "Chỗ ở có nhiều đánh giá gần đây (`number_of_reviews_ltm`) có xu hướng rẻ hơn không?"
   Gợi ý: dùng biểu đồ phân tán, thang log cho giá và điều chỉnh `alpha`.

In [ ]:
# TODO Bài 3 — câu 3 làm mẫu:
s3 = df.dropna(subset=["price"])
s3 = s3[s3["price"] > 0].sample(5000, random_state=2)
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.scatter(s3["number_of_reviews_ltm"], s3["price"], s=5, alpha=0.3, color=XANH)
ax.set_yscale("log")
ax.set_xlabel("số đánh giá trong 12 tháng qua"); ax.set_ylabel("giá (CLP, log)")
ax.set_title("Phòng bận rộn hiếm khi là phòng đắt nhất", loc="left", fontweight="bold")
plt.tight_layout(); plt.show()

## 8. Bài tập về nhà — Bốn hình đầu tiên cho bài tập lớn

Với thành phố của nhóm, hãy tạo bốn hình: (1) chuỗi thời gian đánh giá qua nhiều mốc chụp,
(2) biểu đồ thanh xếp hạng quận theo một chỉ số, (3) phân phối giá trên thang log và (4) biểu đồ phân tán theo toạ độ.
Mỗi hình phải qua danh sách kiểm tra bốn điểm, được lưu vào `figures/` và có hai câu diễn giải trong cell Markdown
ngay dưới hình. Nộp notebook + 4 file PNG.

In [ ]:
RUN_CHALLENGE = False
if RUN_CHALLENGE:
    CITY_BASE = "https://data.insideairbnb.com/..."   # thành phố nhóm bạn
    ...

---

## Tóm tắt buổi học

| Nội dung chính | Vì sao quan trọng |
|---|---|
| Dạng biểu đồ phải phù hợp với câu hỏi | Thể hiện đúng cấu trúc cần phân tích |
| `fig, ax` + tiêu đề thông điệp + đơn vị + nguồn | Hình có thể tự giải thích trong báo cáo |
| Màu nhấn có chủ đích; thanh ngang cho tên dài; `alpha` cho nhiều điểm | Làm nổi bật dữ liệu thay vì trang trí |
| Biểu đồ cột bắt đầu từ 0; hình so sánh dùng chung thang | Tránh gây hiểu sai bằng thị giác |
| `rcParams` + `savefig` vào `figures/` từ pipeline | Phong cách đồng bộ, tái lập được |

**Buổi sau:** seaborn cho biểu đồ thống kê nhiều chiều, bản đồ có ranh giới quận và cách
phản biện biểu đồ do AI sinh ra.